In [ ]:
import os
import sys

import cmcrameri.cm as cmc
import h5py
import hdf5plugin  # noqa: F401 - registers HDF5 compression filters
import matplotlib.pyplot as plt
import numpy as np
import torch
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

# Add project root to path for imports
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from utils.ptychi_utils import unwrap_phase_2d

%matplotlib inline

In [ ]:
output_dir = '/mnt/localdisk/scratch/aileenluo/fig2/scan478_lsqml'
object_name = 'scan478_lsqml_jitter_epoch3000'
with h5py.File(os.path.join(output_dir, object_name + '_para.hdf5'), 'r') as f:
    print("Objects in the HDF5 file:")
    for key in f:
        print(f"- {key}")
    obj = f['object'][:]
    pixel_size_m = f['object'].attrs['pixel_height_m']
    probe = f['probe'][:]
    probe_position_indices = f['probe_position_indexes'][:]
    probe_position_x_m = f['probe_position_x_m'][:]
    probe_position_y_m = f['probe_position_y_m'][:]
    #normalization = f['normalization'][...]

print("Probe shape:", probe.shape, "| Object shape:", obj.shape)
pix_per_um = 1e-6 / pixel_size_m

## Object

In [ ]:
f, ax = plt.subplots(figsize=(7, 3), ncols=2, dpi=300)

amp = ax[0].imshow(np.abs(obj[0])[100:-100, 100:-100], interpolation='none', cmap=cmc.oslo, vmin=0.4, vmax=1.6)
divider = make_axes_locatable(ax[0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(amp, cax=cax, orientation='vertical')
ax[0].set_xticks([])
ax[0].set_yticks([])
ax[0].set_title('Amplitude')

phi = ax[1].imshow(np.angle(obj[0])[100:-100, 100:-100], interpolation='none', cmap=cmc.grayC, vmin=-2.7, vmax=2.7)
divider = make_axes_locatable(ax[1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(phi, cax=cax, orientation='vertical')
scalebar = AnchoredSizeBar(ax[1].transData, 2*pix_per_um, r'$2 \mu m$', 'lower right', pad=0.1, color='#E8E8E8', frameon=False, size_vertical=5)
ax[1].add_artist(scalebar)
ax[1].set_xticks([])
ax[1].set_yticks([])
ax[1].set_title('Phase')

#f.savefig("130nmFOV.svg", transparent=True, bbox_inches="tight", dpi=300)

## Probe

In [ ]:
# Plot first 6 incoherent probe modes (amplitude)

f, ax = plt.subplots(figsize=(7, 1.5), ncols=6, dpi=300)

p0 = ax[0].imshow(np.abs(probe[0, 0]), interpolation='none', cmap=cmc.lajolla)
divider = make_axes_locatable(ax[0])
cax = divider.append_axes('bottom', size='5%', pad=0.05)
f.colorbar(p0, cax=cax, orientation='horizontal')
ax[0].set_xticks([])
ax[0].set_yticks([])

p1 = ax[1].imshow(np.abs(probe[0, 1]), interpolation='none', cmap=cmc.lajolla)
divider = make_axes_locatable(ax[1])
cax = divider.append_axes('bottom', size='5%', pad=0.05)
f.colorbar(p1, cax=cax, orientation='horizontal')
ax[1].set_xticks([])
ax[1].set_yticks([])

p2 = ax[2].imshow(np.abs(probe[0, 2]), interpolation='none', cmap=cmc.lajolla)
divider = make_axes_locatable(ax[2])
cax = divider.append_axes('bottom', size='5%', pad=0.05)
f.colorbar(p2, cax=cax, orientation='horizontal')
ax[2].set_xticks([])
ax[2].set_yticks([])

p3 = ax[3].imshow(np.abs(probe[0, 3]), interpolation='none', cmap=cmc.lajolla)
divider = make_axes_locatable(ax[3])
cax = divider.append_axes('bottom', size='5%', pad=0.05)
f.colorbar(p3, cax=cax, orientation='horizontal')
ax[3].set_xticks([])
ax[3].set_yticks([])

p4 = ax[4].imshow(np.abs(probe[0, 4]), interpolation='none', cmap=cmc.lajolla)
divider = make_axes_locatable(ax[4])
cax = divider.append_axes('bottom', size='5%', pad=0.05)
f.colorbar(p4, cax=cax, orientation='horizontal')
ax[4].set_xticks([])
ax[4].set_yticks([])

p5 = ax[5].imshow(np.abs(probe[0, 5]), interpolation='none', cmap=cmc.lajolla)
divider = make_axes_locatable(ax[5])
cax = divider.append_axes('bottom', size='5%', pad=0.05)
f.colorbar(p5, cax=cax, orientation='horizontal')
ax[5].set_xticks([])
ax[5].set_yticks([])

#f.savefig("flyscan_lsqml_probe_amp.svg", transparent=True, bbox_inches="tight", dpi=300)

In [ ]:
# Fit probe FWHM
from scipy.optimize import curve_fit


def gaussian_2d(coords, amp, x0, y0, sigma_x, sigma_y, theta, offset):
    x, y = coords
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)
    x_shift = x - x0
    y_shift = y - y0
    x_rot = cos_t * x_shift + sin_t * y_shift
    y_rot = -sin_t * x_shift + cos_t * y_shift
    return offset + amp * np.exp(-0.5 * ((x_rot / sigma_x) ** 2 + (y_rot / sigma_y) ** 2)).ravel()


def fit_probe_fwhm(mode=0):
    image = np.abs(probe[0, mode]).astype(float)
    y, x = np.indices(image.shape)
    offset0 = np.percentile(image, 5)
    weights = np.clip(image - offset0, 0, None)
    if np.sum(weights) == 0:
        weights = image

    x0 = np.sum(x * weights) / np.sum(weights)
    y0 = np.sum(y * weights) / np.sum(weights)
    sigma_x0 = np.sqrt(np.sum(((x - x0) ** 2) * weights) / np.sum(weights))
    sigma_y0 = np.sqrt(np.sum(((y - y0) ** 2) * weights) / np.sum(weights))
    p0 = (image.max() - offset0, x0, y0, sigma_x0, sigma_y0, 0.0, offset0)
    bounds = (
        (0, 0, 0, 0.1, 0.1, -np.pi / 2, -np.inf),
        (np.inf, image.shape[1] - 1, image.shape[0] - 1, image.shape[1], image.shape[0], np.pi / 2, np.inf),
    )

    popt, pcov = curve_fit(
        gaussian_2d,
        (x, y),
        image.ravel(),
        p0=p0,
        bounds=bounds,
        maxfev=20000,
    )
    _, x0, y0, sigma_x, sigma_y, _, _ = popt
    fwhm_x = 2 * np.sqrt(2 * np.log(2)) * sigma_x
    fwhm_y = 2 * np.sqrt(2 * np.log(2)) * sigma_y
    return {
        "mode": mode,
        "fit": popt,
        "covariance": pcov,
        "fwhm_x_px": fwhm_x,
        "fwhm_y_px": fwhm_y,
        "fwhm_geometric_mean_px": np.sqrt(fwhm_x * fwhm_y),
    }


probe_fwhm = [fit_probe_fwhm(mode) for mode in (0, 1)]
for result in probe_fwhm:
    line = (
        f"mode {result['mode']}: "
        f"FWHM_x={result['fwhm_x_px']:.2f} px, "
        f"FWHM_y={result['fwhm_y_px']:.2f} px, "
        f"geomean={result['fwhm_geometric_mean_px']:.2f} px"
    )
    if "pixel_size" in globals():
        line += (
            f" | FWHM_x={result['fwhm_x_px'] * pixel_size_m * 1e9:.1f} nm, "
            f"FWHM_y={result['fwhm_y_px'] * pixel_size_m * 1e9:.1f} nm"
        )
    print(line)

f, ax = plt.subplots(figsize=(6, 3), ncols=2)
for axis, result in zip(ax, probe_fwhm):
    mode = result["mode"]
    image = np.abs(probe[0, mode]).astype(float)
    y, x = np.indices(image.shape)
    fit_image = gaussian_2d((x, y), *result["fit"]).reshape(image.shape)
    axis.imshow(image, interpolation="none")
    axis.contour(fit_image, levels=[result["fit"][6] + 0.5 * result["fit"][0]], colors="white", linewidths=0.8)
    axis.set_title(f"mode {mode}: {result['fwhm_geometric_mean_px']:.1f} px")
    axis.set_xticks([])
    axis.set_yticks([])

In [ ]:
from utils.math import estimate_scan_overlap

positions_m = np.column_stack([probe_position_y_m, probe_position_x_m])

# Approximate the elliptical primary probe by the arithmetic mean of its fitted axes.
primary_probe_fwhm_px = np.mean([
    probe_fwhm[0]["fwhm_x_px"],
    probe_fwhm[0]["fwhm_y_px"],
])
primary_probe_fwhm_m = 78.18 * pixel_size_m #primary_probe_fwhm_px * pixel_size_m

scan_overlap = estimate_scan_overlap(
    positions_m,
    probe_fwhm=primary_probe_fwhm_m,
    k=1,
)

print(
    f"Average primary-probe FWHM: {primary_probe_fwhm_px:.2f} px "
    f"({primary_probe_fwhm_m * 1e9:.1f} nm)"
)
print(f"Mean nearest-neighbor spacing: {scan_overlap['mean_spacing'] * 1e9:.1f} nm")
print(f"Mean overlap: {100 * scan_overlap['mean_overlap']:.1f}%")
print(f"Median overlap: {100 * scan_overlap['median_overlap']:.1f}%")
print(
    f"10th-90th percentile overlap: "
    f"{100 * scan_overlap['overlap_p10']:.1f}%-"
    f"{100 * scan_overlap['overlap_p90']:.1f}%"
)

## Optional for diagnostics

### Total intensity as a function of position

In [ ]:
with h5py.File(os.path.join(output_dir, object_name + '_dp.hdf5'), 'r') as f:
    print("Objects in the HDF5 file:")
    for key in f:
        print(f"- {key}")
    data = f['dp'][:]
data.shape, data.dtype

In [ ]:
f, ax = plt.subplots(figsize=(3, 3))

raw = ax.scatter(positions_m[:, 1], positions_m[:, 0], c=data.sum(axis=(1, 2)), cmap='viridis', s=5)
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(raw, cax=cax, orientation='vertical')
ax.set_title('Total intensity')

### Unwrap phase if phase wrap present

In [ ]:
phase = unwrap_phase_2d(torch.from_numpy(obj[0]))

### Polynomial phase ramp

In [ ]:
def remove_polynomial_background(
    img,
    flat_region_mask,
    polyfit_order: int = 1,
):
    """
    Fit a 2D polynomial to the region that is supposed to be flat in an
    image, and subtract the fitted function from the image.

    Parameters
    ----------
    img : Tensor
        The input image.
    flat_region_mask : Tensor
        A boolean mask with the same shape as `img` that specifies the region of
        the image that should be flat.
    polyfit_order : int, optional
        The order of the polynomial to fit. Should be an integer >= 0. If 0,
        just subtract the average. The default is 1.

    Returns
    -------
    Tensor
        The image with the polynomial background subtracted.
    """
    if polyfit_order == 0:
        return img - img[flat_region_mask].mean()

    if polyfit_order < 0:
        raise ValueError("polyfit_order must be >= 0.")

    ys, xs = torch.where(flat_region_mask)
    if ys.numel() == 0:
        raise ValueError("flat_region_mask must contain at least one True value.")

    y_full, x_full = torch.meshgrid(
        torch.arange(img.shape[0]), torch.arange(img.shape[1]), indexing="ij"
    )
    y_full = y_full.reshape(-1)
    x_full = x_full.reshape(-1)

    fit_dtype = torch.float64
    ys = ys.to(device=img.device, dtype=fit_dtype)
    xs = xs.to(device=img.device, dtype=fit_dtype)
    y_full = y_full.to(device=img.device, dtype=fit_dtype)
    x_full = x_full.to(device=img.device, dtype=fit_dtype)

    y_scale = max(img.shape[0] - 1, 1)
    x_scale = max(img.shape[1] - 1, 1)
    ys = ys / y_scale
    xs = xs / x_scale
    y_full = y_full / y_scale
    x_full = x_full / x_scale

    basis = []
    basis_full = []
    for total_order in range(polyfit_order + 1):
        for y_order in range(total_order + 1):
            x_order = total_order - y_order
            basis.append((ys**y_order) * (xs**x_order))
            basis_full.append((y_full**y_order) * (x_full**x_order))

    a_mat = torch.stack(basis, dim=1)
    b_vec = img[flat_region_mask].reshape(-1, 1).to(fit_dtype)
    b_mean = b_vec.mean()
    b_std = b_vec.std()
    if b_std == 0:
        b_std = torch.tensor(1.0, dtype=b_vec.dtype, device=b_vec.device)
    b_vec_std = (b_vec - b_mean) / b_std
    x_vec = torch.linalg.lstsq(a_mat, b_vec_std).solution

    a_mat_full = torch.stack(basis_full, dim=1)
    bg = a_mat_full @ x_vec

    bg = bg * b_std + b_mean
    bg = bg.reshape(img.shape).to(img.dtype)

    return img - bg